# 🎨 [컴비 2조] 가짜 생성 C (SDXL) - 24종 물건별 맞춤 제작 스튜디오
**실행 방법:** 상단 메뉴 `런타임` > `런타임 유형 변경` > **T4 GPU** 선택 후 실행하세요.


### 📌 [1번 셀] 필수 라이브러리 설치 및 SDXL 모델 로드


In [ ]:
!pip install -q diffusers transformers accelerate safetensors

import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

MODEL_ID = "RunDiffusion/Juggernaut-XL-v9"
MODEL_NAME = "Juggernaut-XL-v9"

# fp16 충돌 방지 전용 VAE 로드
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16
)

try:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        vae=vae,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True
    ).to("cuda")
except Exception:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        vae=vae,
        torch_dtype=torch.float16,
        use_safetensors=True
    ).to("cuda")

print(f"✅ {MODEL_NAME} 모델 로드 완료!")


### 📌 [2번 셀] 핵심 프롬프트 템플릿 및 generate() 함수 정의


In [ ]:
import os, csv, random, shutil
import torch
from datetime import datetime
from IPython.display import display

OUT_DIR = "fakeC"
IMG_DIR = os.path.join(OUT_DIR, "C")
TEST_DIR = os.path.join(OUT_DIR, "test")
CSV_PATH = os.path.join(OUT_DIR, "fake_mapping.csv")

os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

FAKE_MAPPING_COLUMNS = [
    "fake_filename", "fake_type", "source_filename", "mask_filename", "object",
    "prompt", "negative_prompt", "seed", "model", "created_by", "is_reinforcement",
]

DATES = ["2026.11.17", "2026.11.18", "2026.11.19", "2026.11.20", "2026.11.21"]

# 토큰 길이 검증 함수 (75토큰 이하여야 뒤가 안 잘림)
def tok_len(s):
    return len(pipe.tokenizer(s).input_ids) - 2

# 팀 최적화 초경량 네거티브 베이스 (39토큰으로 압축하여 77토큰 잘림 완벽 방지)
NEGATIVE_BASE = (
    "two sticky notes, stack of papers, glowing screen, text on screen, cartoon, "
    "3d render, gibberish, deformed, watermark, logo, cable, close-up, cropped"
)
NEGATIVE_NOMEMO = "sticky note, memo, paper, note, handwriting, text"

# 기본 프롬프트 템플릿 (메모 있음 / 메모 없음)
TEMPLATE = (
    'a casual snapshot of a {obj} on a {bg}, '
    'a single {paper} with handwritten date "{date}" placed right next to it, clearly visible, '
    '{angle}, {light}, realistic, authentic amateur photo, clean shot'
)
TEMPLATE_NOMEMO = (
    'a casual snapshot of a {obj} placed on a {bg}, '
    '{angle}, {light}, realistic, authentic amateur photo, clean shot'
)

def make_fake_c_filename(obj_key, idx, seed, has_memo=True, is_test=False):
    prefix = "test_" if is_test else ""
    if has_memo:
        return f"{prefix}fakeC_{obj_key}_{idx:03d}_s{seed}.png"
    return f"{prefix}fakeC_{obj_key}_nomemo_{idx:03d}_s{seed}.png"

def generate(
    obj, bg, light,
    paper="small yellow sticky note",
    angle="angled 45-degree view",
    date="2026.11.18",
    seed=None,
    idx=1,
    steps=30,
    guidance=7.5,
    extra_negative="",
    obj_key="unknown",
    has_memo=True,
    is_test=False,
    record_log=True,
    created_by="changjun",
    is_reinforcement=0,
):
    seed = seed if seed is not None else random.randint(1, 99999)
    if has_memo:
        prompt = TEMPLATE.format(obj=obj, bg=bg, light=light, paper=paper, angle=angle, date=date)
        neg_parts = [NEGATIVE_BASE, extra_negative]
    else:
        prompt = TEMPLATE_NOMEMO.format(obj=obj, bg=bg, light=light, angle=angle, date=date)
        neg_parts = [NEGATIVE_BASE, NEGATIVE_NOMEMO, extra_negative]

    neg_prompt = ", ".join(p.strip(", ") for p in neg_parts if p and p.strip(", "))
    p_len = tok_len(prompt)
    n_len = tok_len(neg_prompt)
    print(f"토큰 수 → positive {p_len} / negative {n_len} (75 이하여야 함)")
    if p_len > 75:
        print(f"⚠️ 경고: 프롬프트가 {p_len}토큰으로 75토큰을 초과했습니다. 뒤쪽 단어가 잘릴 수 있습니다.")
    if n_len > 75:
        print(f"⚠️ 경고: 네거티브가 {n_len}토큰으로 75토큰을 초과했습니다. 뒤쪽 단어가 잘릴 수 있습니다.")

    g = torch.Generator("cuda").manual_seed(seed)

    img = pipe(
        prompt,
        negative_prompt=neg_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=g
    ).images[0]

    fname = make_fake_c_filename(obj_key, idx, seed, has_memo=has_memo, is_test=is_test)
    save_dir = TEST_DIR if is_test else IMG_DIR
    img.save(os.path.join(save_dir, fname))

    if record_log and not is_test:
        new_file = not os.path.exists(CSV_PATH)
        with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            if new_file:
                w.writerow(FAKE_MAPPING_COLUMNS)
            w.writerow([
                fname, "C", "", "", obj_key,
                prompt, neg_prompt, seed, MODEL_NAME, created_by, is_reinforcement,
            ])

    return img

print("✅ 최신 팀 규약 적용 완료! (파일명 fakeC_{물건}_{번호}_s{seed}.png / nomemo_ 지원 / fake_mapping.csv)")


## 🔍 [3번 섹션] 24종 물건별 구도 확인 (3-1 ~ 3-24 연속)
먼저 3-1부터 3-24까지 3장씩 돌려 구도를 확인하세요.


#### 📌 [3-1] 휴대폰 (smartphone) 구도 확인 (3장)


In [ ]:
# [3-1] 휴대폰 (smartphone) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='black smartphone lying flat, screen off',
        bg='light wooden desk',
        light='soft window light',
        paper='small yellow sticky note',
        angle='top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='smartphone',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='pencil, pen, bowl, cloth, feature phone, keypad, two phones',
    )
    print(f"[{i+1}/3] 휴대폰 완료")
    display(img)

#### 📌 [3-2] 노트북 (laptop) 구도 확인 (3장)


In [ ]:
# [3-2] 노트북 (laptop) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single silver laptop with powered off dark blank screen',
        bg='empty wooden desk',
        light='natural window sunlight',
        paper='small yellow sticky note',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='laptop',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='coffee, cup, mug, clutter, display text, numbers on screen, wallpaper, two laptops',
    )
    print(f"[{i+1}/3] 노트북 완료")
    display(img)


#### 📌 [3-3] 워치 (smartwatch) 구도 확인 (3장)


In [ ]:
# [3-3] 워치 (smartwatch) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single black smartwatch with black silicone strap',
        bg='dark wooden desk',
        light='soft window light',
        paper='small white paper note',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='smartwatch',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=8.0,
        extra_negative='analog watch, metal watch, pen, pencil, two watches, numbers on watch display, two notes',
    )
    print(f"[{i+1}/3] 워치 완료")
    display(img)


#### 📌 [3-4] 이어폰 (earbuds) 구도 확인 (3장)


In [ ]:
# [3-4] 이어폰 (earbuds) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single black wireless earbuds charging case, open',
        bg='wooden desk',
        light='soft window light',
        paper='small white paper note',
        angle='top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='earbuds',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=8.0,
        extra_negative='smartphone, phone, pen, pencil, lighter, two cases, two notes, clutter',
    )
    print(f"[{i+1}/3] 이어폰 완료")
    display(img)


#### 📌 [3-5] 헤드셋 (headphones) 구도 확인 (3장)


In [ ]:
# [3-5] 헤드셋 (headphones) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single pair of black over-ear headphones lying flat',
        bg='empty wooden desk',
        light='soft window light',
        paper='small yellow sticky note',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='headphones',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='two headphones, multiple headphones, cup, mug, coffee, stand, clutter',
    )
    print(f"[{i+1}/3] 헤드셋 완료")
    display(img)


#### 📌 [3-6] 키보드 (keyboard) 구도 확인 (3장)


In [ ]:
# [3-6] 키보드 (keyboard) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single black mechanical keyboard alone on an empty desk',
        bg='empty wooden study desk',
        light='natural window sunlight',
        paper='small yellow sticky note in front of spacebar',
        angle='angled 45-degree top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='keyboard',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 키보드 완료")
    display(img)

#### 📌 [3-7] 마우스 (mouse) 구도 확인 (3장)


In [ ]:
# [3-7] 마우스 (mouse) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single black wireless computer mouse alone on an empty desk',
        bg='empty wooden desk',
        light='natural window sunlight',
        paper='small yellow sticky note beside mouse',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='mouse',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 마우스 완료")
    display(img)

#### 📌 [3-8] 신발 (sneakers) 구도 확인 (3장)


In [ ]:
# [3-8] 신발 (sneakers) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='pair of casual sneakers',
        bg='wooden floor',
        light='natural window sunlight',
        paper='small yellow sticky note on the floor in front',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='sneakers',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='shoebox, box, wrapping paper, tissue paper, feet, legs, clutter',
    )
    print(f"[{i+1}/3] 신발 완료")
    display(img)


#### 📌 [3-9] 텀블러 (tumbler) 구도 확인 (3장)


In [ ]:
# [3-9] 텀블러 (tumbler) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single stainless steel tumbler standing upright',
        bg='empty light wooden desk',
        light='bright indoor fluorescent light',
        paper='small yellow sticky note',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='tumbler',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.5,
        extra_negative='two tumblers, multiple cups, pen, pencil, notebook, laptop, computer, plant, text on tumbler, clutter',
    )
    print(f"[{i+1}/3] 텀블러 완료")
    display(img)


#### 📌 [3-10] 옷 (hoodie) 구도 확인 (3장)


In [ ]:
# [3-10] 옷 (hoodie) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single neatly folded plain gray hoodie',
        bg='plain white bed sheet',
        light='bright indoor fluorescent light',
        paper='small yellow sticky note',
        angle='top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='hoodie',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.5,
        extra_negative='text on clothes, graphic print, logo, multiple notes, extra papers, pen, pencil, clutter',
    )
    print(f"[{i+1}/3] 옷 완료")
    display(img)


#### 📌 [3-11] 의자 (chair) 구도 확인 (3장)


In [ ]:
# [3-11] 의자 (chair) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='black swivel desk chair with a white paper sheet lying on its seat',
        bg='wooden floor',
        light='bright indoor fluorescent light',
        paper='white paper sheet',
        angle='wide shot, angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='chair',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='monitor, multiple chairs, person, clutter, pen',
    )
    print(f"[{i+1}/3] 의자 완료")
    display(img)


#### 📌 [3-12] 지갑 (wallet) 구도 확인 (3장)


In [ ]:
# [3-12] 지갑 (wallet) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single brown leather bifold wallet lying flat, closed',
        bg='empty light wooden desk',
        light='soft window light',
        paper='bright yellow sticky note',
        angle='top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='wallet',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=8.0,
        extra_negative='smartphone, phone, keys, pen, pencil, coins, watch, glasses, clutter, two wallets',
    )
    print(f"[{i+1}/3] 지갑 완료")
    display(img)


#### 📌 [3-13] 게임컨트롤러 (gamecontroller) 구도 확인 (3장)


In [ ]:
# [3-13] 게임컨트롤러 (gamecontroller) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='wireless game controller',
        bg='wooden desk',
        light='soft window light',
        paper='small yellow sticky note beside the game controller',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='gamecontroller',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 게임컨트롤러 완료")
    display(img)

#### 📌 [3-14] 가방 (backpack) 구도 확인 (3장)


In [ ]:
# [3-14] 가방 (backpack) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='everyday backpack sitting upright',
        bg='clean floor against wall',
        light='natural window sunlight',
        paper='small yellow sticky note attached to the front pocket',
        angle='eye-level front 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='backpack',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 가방 완료")
    display(img)

#### 📌 [3-15] 책 (book) 구도 확인 (3장)


In [ ]:
# [3-15] 책 (book) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single closed dark blue hardcover book lying flat',
        bg='empty light wooden desk',
        light='soft window light',
        paper='bright yellow sticky note',
        angle='top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='book',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.5,
        extra_negative='title text, cup, mug, coffee, glasses, phone, camera, stack of books, clutter',
    )
    print(f"[{i+1}/3] 책 완료")
    display(img)


#### 📌 [3-16] 화장품 (cosmetics) 구도 확인 (3장)


In [ ]:
# [3-16] 화장품 (cosmetics) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single white skincare lotion pump bottle lying on the left side',
        bg='dark wooden desk',
        light='soft window light',
        paper='small white paper note',
        angle='top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='cosmetics',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=6.5,
        extra_negative='label, text on bottle, brand logo, thick notepad, stack of notes, pen, two notes, clutter',
    )
    print(f"[{i+1}/3] 화장품 완료")
    display(img)


#### 📌 [3-17] 냉장고 (fridge) 구도 확인 (3장)


In [ ]:
# [3-17] 냉장고 (fridge) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single one-door silver mini fridge with one white note taped flat on its front door',
        bg='plain tile floor',
        light='bright indoor fluorescent light',
        paper='white paper note',
        angle='wide shot, eye-level front view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='fridge',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=6.5,
        extra_negative='washing machine, water dispenser, plant, microwave, sink, multiple appliances, two notes, two doors, clutter',
    )
    print(f"[{i+1}/3] 냉장고 완료")
    display(img)


#### 📌 [3-18] 삼각대 (tripod) 구도 확인 (3장)


In [ ]:
# [3-18] 삼각대 (tripod) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single black aluminum three-legged tripod standing upright',
        bg='empty wooden floor',
        light='natural window sunlight',
        paper='bright yellow sticky note',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='tripod',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.5,
        extra_negative='camera, dslr, lens, phone, thick notepad, stack of notes, pen, pencil, two tripods, clutter',
    )
    print(f"[{i+1}/3] 삼각대 완료")
    display(img)


#### 📌 [3-19] 도시락 (lunchbox) 구도 확인 (3장)


In [ ]:
# [3-19] 도시락 (lunchbox) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single closed plastic bento lunchbox container',
        bg='empty light wooden desk',
        light='bright indoor fluorescent light',
        paper='single thin flat yellow sticky note sheet',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='lunchbox',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.5,
        extra_negative='thick pad, stack of sticky notes, notepad block, cube, chopsticks, food, cup, pen, pencil, clutter',
    )
    print(f"[{i+1}/3] 도시락 완료")
    display(img)


#### 📌 [3-20] 책상 (desk) 구도 확인 (3장)


In [ ]:
# [3-20] 책상 (desk) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='wooden study desk with a large white A4 paper lying on top of the desk',
        bg='wooden floor',
        light='natural window sunlight',
        paper='white A4 paper',
        angle='wide shot, angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='desk',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=6.5,
        extra_negative='chair, laptop, monitor, books, clutter, two desks, two notes',
    )
    print(f"[{i+1}/3] 책상 완료")
    display(img)


#### 📌 [3-21] 마이크 (microphone) 구도 확인 (3장)


In [ ]:
# [3-21] 마이크 (microphone) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='desktop microphone on a small stand',
        bg='wooden desk',
        light='soft window light',
        paper='small yellow sticky note beside the microphone base',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='microphone',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 마이크 완료")
    display(img)

#### 📌 [3-22] 목도리 (muffler) 구도 확인 (3장)


In [ ]:
# [3-22] 목도리 (muffler) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='folded dark navy knitted scarf with fringed ends and a bright yellow sticky note on it',
        bg='light wooden floor',
        light='soft window light',
        paper='bright yellow sticky note',
        angle='top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='muffler',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=6.5,
        extra_negative='person, neck, blanket, sweater, pen, pencil, two notes, clutter',
    )
    print(f"[{i+1}/3] 목도리 완료")
    display(img)


#### 📌 [3-23] 장갑 (gloves) 구도 확인 (3장)


In [ ]:
# [3-23] 장갑 (gloves) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='pair of winter gloves',
        bg='wooden table',
        light='soft window light',
        paper='small yellow sticky note beside the gloves',
        angle='top-down view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='gloves',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=7.0,
        extra_negative='',
    )
    print(f"[{i+1}/3] 장갑 완료")
    display(img)

#### 📌 [3-24] 향수 (perfume) 구도 확인 (3장)


In [ ]:
# [3-24] 향수 (perfume) 구도 확인 (3장 - 테스트 폴더 저장)
for i in range(3):
    img = generate(
        obj='single opaque black matte glass perfume bottle with spray cap',
        bg='dark wooden desk',
        light='soft window light',
        paper='small white paper note',
        angle='angled 45-degree view',
        date='2026.11.18',
        seed=random.randint(1, 99999),
        idx=i + 1,
        obj_key='perfume',
        is_test=True,
        record_log=False,
        steps=35,
        guidance=6.5,
        extra_negative='yellow liquid, clear bottle, label, text on bottle, brand logo, thick pad, stack of notes, pen, two notes, clutter',
    )
    print(f"[{i+1}/3] 향수 완료")
    display(img)


## 🚀 [4번 섹션] 24종 물건별 15장 본 양산 (메모 있음 12장 + 메모 없음 3장, 4-1 ~ 4-24)


#### 🚀 [4-1] 휴대폰 (smartphone) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-1] 휴대폰 (smartphone) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_smartphone_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='black smartphone lying flat, screen off',
        bg='light wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='pencil, pen, bowl, cloth, feature phone, keypad, two phones',
        obj_key='smartphone',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 휴대폰 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_smartphone_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='black smartphone lying flat, screen off',
        bg='light wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='pencil, pen, bowl, cloth, feature phone, keypad, two phones',
        obj_key='smartphone',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 휴대폰 완료")
    display(img)

print("🎉 [휴대폰] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-2] 노트북 (laptop) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-2] 노트북 (laptop) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_laptop_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='single silver laptop with powered off dark blank screen',
        bg='empty wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='coffee, cup, mug, clutter, display text, numbers on screen, wallpaper, two laptops',
        obj_key='laptop',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 노트북 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_laptop_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single silver laptop with powered off dark blank screen',
        bg='empty wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='coffee, cup, mug, clutter, display text, numbers on screen, wallpaper, two laptops',
        obj_key='laptop',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 노트북 완료")
    display(img)

print("🎉 [노트북] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-3] 워치 (smartwatch) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-3] 워치 (smartwatch) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'top-down view', 'angled 45-degree view']
PAPER_BASE = 'small white paper note'

# 1) 메모 있는 사진 12장 (fakeC_smartwatch_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    img = generate(
        obj='single black smartwatch with black silicone strap',
        bg='dark wooden desk',
        light=random.choice(lights),
        paper=PAPER_BASE,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=8.0,
        extra_negative='analog watch, metal watch, pen, pencil, two watches, numbers on watch display, two notes',
        obj_key='smartwatch',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 워치 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_smartwatch_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single black smartwatch with black silicone strap',
        bg='dark wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=8.0,
        extra_negative='analog watch, metal watch, pen, pencil, two watches, numbers on watch display, two notes',
        obj_key='smartwatch',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 워치 완료")
    display(img)

print("🎉 [워치] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-4] 이어폰 (earbuds) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-4] 이어폰 (earbuds) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'top-down view']
PAPER_BASE = 'small white paper note'

# 1) 메모 있는 사진 12장 (fakeC_earbuds_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    img = generate(
        obj='single black wireless earbuds charging case, open',
        bg='wooden desk',
        light=random.choice(lights),
        paper=PAPER_BASE,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=8.0,
        extra_negative='smartphone, phone, pen, pencil, lighter, two cases, two notes, clutter',
        obj_key='earbuds',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 이어폰 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_earbuds_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single black wireless earbuds charging case, open',
        bg='wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=8.0,
        extra_negative='smartphone, phone, pen, pencil, lighter, two cases, two notes, clutter',
        obj_key='earbuds',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 이어폰 완료")
    display(img)

print("🎉 [이어폰] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-5] 헤드셋 (headphones) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-5] 헤드셋 (headphones) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_headphones_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='single pair of black over-ear headphones lying flat',
        bg='empty wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='two headphones, multiple headphones, cup, mug, coffee, stand, clutter',
        obj_key='headphones',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 헤드셋 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_headphones_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single pair of black over-ear headphones lying flat',
        bg='empty wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='two headphones, multiple headphones, cup, mug, coffee, stand, clutter',
        obj_key='headphones',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 헤드셋 완료")
    display(img)

print("🎉 [헤드셋] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-6] 키보드 (keyboard) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-6] 키보드 (keyboard) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note in front of spacebar'

# 1) 메모 있는 사진 12장 (fakeC_keyboard_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='single black mechanical keyboard alone on an empty desk',
        bg='empty wooden study desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='mouse, mousepad, laptop, monitor, screen, two keyboards, multiple keyboards, cup, mug, plant, clutter',
        obj_key='keyboard',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 키보드 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_keyboard_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single black mechanical keyboard alone on an empty desk',
        bg='empty wooden study desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='mouse, mousepad, laptop, monitor, screen, two keyboards, multiple keyboards, cup, mug, plant, clutter',
        obj_key='keyboard',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 키보드 완료")
    display(img)

print("🎉 [키보드] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-7] 마우스 (mouse) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-7] 마우스 (mouse) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note beside mouse'

# 1) 메모 있는 사진 12장 (fakeC_mouse_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='single black wireless computer mouse alone on an empty desk',
        bg='empty wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='keyboard, keycaps, laptop, monitor, screen, two mice, multiple mice, cup, mug, plant, clutter',
        obj_key='mouse',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 마우스 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_mouse_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single black wireless computer mouse alone on an empty desk',
        bg='empty wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='keyboard, keycaps, laptop, monitor, screen, two mice, multiple mice, cup, mug, plant, clutter',
        obj_key='mouse',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 마우스 완료")
    display(img)

print("🎉 [마우스] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-8] 신발 (sneakers) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-8] 신발 (sneakers) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note on the floor in front'

# 1) 메모 있는 사진 12장 (fakeC_sneakers_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='pair of casual sneakers',
        bg='wooden floor',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='shoebox, box, wrapping paper, tissue paper, feet, legs, clutter',
        obj_key='sneakers',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 신발 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_sneakers_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='pair of casual sneakers',
        bg='wooden floor',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='shoebox, box, wrapping paper, tissue paper, feet, legs, clutter',
        obj_key='sneakers',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 신발 완료")
    display(img)

print("🎉 [신발] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-9] 텀블러 (tumbler) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-9] 텀블러 (tumbler) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'angled 45-degree view']
PAPER_BASE = 'small yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_tumbler_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='single stainless steel tumbler standing upright',
        bg='empty light wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='two tumblers, multiple cups, pen, pencil, notebook, laptop, computer, plant, text on tumbler, clutter',
        obj_key='tumbler',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 텀블러 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_tumbler_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single stainless steel tumbler standing upright',
        bg='empty light wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='two tumblers, multiple cups, pen, pencil, notebook, laptop, computer, plant, text on tumbler, clutter',
        obj_key='tumbler',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 텀블러 완료")
    display(img)

print("🎉 [텀블러] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-10] 옷 (hoodie) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-10] 옷 (hoodie) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'angled 45-degree view']
PAPER_BASE = 'small yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_hoodie_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='single neatly folded plain gray hoodie',
        bg='plain white bed sheet',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='text on clothes, graphic print, logo, multiple notes, extra papers, pen, pencil, clutter',
        obj_key='hoodie',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 옷 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_hoodie_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single neatly folded plain gray hoodie',
        bg='plain white bed sheet',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='text on clothes, graphic print, logo, multiple notes, extra papers, pen, pencil, clutter',
        obj_key='hoodie',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 옷 완료")
    display(img)

print("🎉 [옷] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-11] 의자 (chair) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-11] 의자 (chair) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['wide shot, angled 45-degree view', 'wide shot, eye-level front view', 'wide shot, angled 45-degree view']
PAPER_BASE = 'white paper sheet'

# 1) 메모 있는 사진 12장 (fakeC_chair_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    img = generate(
        obj='black swivel desk chair with a white paper sheet lying on its seat',
        bg='wooden floor',
        light=random.choice(lights),
        paper=PAPER_BASE,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='monitor, multiple chairs, person, clutter, pen',
        obj_key='chair',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 의자 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_chair_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single black swivel desk chair',
        bg='wooden floor',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='monitor, multiple chairs, person, clutter, pen',
        obj_key='chair',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 의자 완료")
    display(img)

print("🎉 [의자] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-12] 지갑 (wallet) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-12] 지갑 (wallet) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'top-down view']
PAPER_BASE = 'bright yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_wallet_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='single brown leather bifold wallet lying flat, closed',
        bg='empty light wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=8.0,
        extra_negative='smartphone, phone, keys, pen, pencil, coins, watch, glasses, clutter, two wallets',
        obj_key='wallet',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 지갑 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_wallet_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single brown leather bifold wallet lying flat, closed',
        bg='empty light wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=8.0,
        extra_negative='smartphone, phone, keys, pen, pencil, coins, watch, glasses, clutter, two wallets',
        obj_key='wallet',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 지갑 완료")
    display(img)

print("🎉 [지갑] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-13] 게임컨트롤러 (gamecontroller) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-13] 게임컨트롤러 (gamecontroller) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note beside the game controller'

# 1) 메모 있는 사진 12장 (fakeC_gamecontroller_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='wireless game controller',
        bg='wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='',
        obj_key='gamecontroller',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 게임컨트롤러 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_gamecontroller_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='wireless game controller',
        bg='wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='',
        obj_key='gamecontroller',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 게임컨트롤러 완료")
    display(img)

print("🎉 [게임컨트롤러] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-14] 가방 (backpack) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-14] 가방 (backpack) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note attached to the front pocket'

# 1) 메모 있는 사진 12장 (fakeC_backpack_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='everyday backpack sitting upright',
        bg='clean floor against wall',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='',
        obj_key='backpack',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 가방 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_backpack_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='everyday backpack sitting upright',
        bg='clean floor against wall',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='',
        obj_key='backpack',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 가방 완료")
    display(img)

print("🎉 [가방] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-15] 책 (book) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-15] 책 (book) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'top-down view']
PAPER_BASE = 'bright yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_book_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    # 노란 포스트잇 고정 (흰 종이 전환 시 표지 겹침 방지)
    img = generate(
        obj='single closed dark blue hardcover book lying flat',
        bg='empty light wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='title text, cup, mug, coffee, glasses, phone, camera, stack of books, clutter',
        obj_key='book',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 책 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_book_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single closed dark blue hardcover book lying flat',
        bg='empty light wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='title text, cup, mug, coffee, glasses, phone, camera, stack of books, clutter',
        obj_key='book',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 책 완료")
    display(img)

print("🎉 [책] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-16] 화장품 (cosmetics) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-16] 화장품 (cosmetics) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'top-down view']
PAPER_BASE = 'small yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_cosmetics_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    img = generate(
        obj='single white skincare lotion pump bottle lying on the left side',
        bg='dark wooden desk',
        light=random.choice(lights),
        paper=PAPER_BASE,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='label, text on bottle, brand logo, thick notepad, stack of notes, pen, two notes, clutter',
        obj_key='cosmetics',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 화장품 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_cosmetics_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single white skincare lotion pump bottle',
        bg='dark wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='label, text on bottle, brand logo, thick notepad, stack of notes, pen, two notes, clutter',
        obj_key='cosmetics',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 화장품 완료")
    display(img)

print("🎉 [화장품] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-17] 냉장고 (fridge) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-17] 냉장고 (fridge) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['wide shot, eye-level front view', 'wide shot, angled 45-degree view', 'wide shot, eye-level front view']
PAPER_BASE = 'white paper note'

# 1) 메모 있는 사진 12장 (fakeC_fridge_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    img = generate(
        obj='single one-door silver mini fridge with one white note taped flat on its front door',
        bg='plain tile floor',
        light=random.choice(lights),
        paper=PAPER_BASE,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='washing machine, water dispenser, plant, microwave, sink, multiple appliances, two notes, two doors, clutter',
        obj_key='fridge',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 냉장고 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_fridge_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single one-door silver mini fridge',
        bg='plain tile floor',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='washing machine, water dispenser, plant, microwave, sink, multiple appliances, two notes, two doors, clutter',
        obj_key='fridge',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 냉장고 완료")
    display(img)

print("🎉 [냉장고] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-18] 삼각대 (tripod) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-18] 삼각대 (tripod) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'top-down view']
PAPER_BASE = 'bright yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_tripod_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow" in paper and random.random() < 0.4:
        paper = "single flat white memo paper sheet"
    img = generate(
        obj='single black aluminum three-legged tripod standing upright',
        bg='empty wooden floor',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='camera, dslr, lens, phone, thick notepad, stack of notes, pen, pencil, two tripods, clutter',
        obj_key='tripod',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 삼각대 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_tripod_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single black aluminum three-legged tripod standing upright',
        bg='empty wooden floor',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='camera, dslr, lens, phone, thick notepad, stack of notes, pen, pencil, two tripods, clutter',
        obj_key='tripod',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 삼각대 완료")
    display(img)

print("🎉 [삼각대] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-19] 도시락 (lunchbox) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-19] 도시락 (lunchbox) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'top-down view', 'angled 45-degree view']
PAPER_BASE = 'single thin flat yellow sticky note sheet'

# 1) 메모 있는 사진 12장 (fakeC_lunchbox_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow" in paper and random.random() < 0.4:
        paper = "single flat white memo paper sheet"
    img = generate(
        obj='single closed plastic bento lunchbox container',
        bg='empty light wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='thick pad, stack of sticky notes, notepad block, cube, chopsticks, food, cup, pen, pencil, clutter',
        obj_key='lunchbox',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 도시락 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_lunchbox_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single closed plastic bento lunchbox container',
        bg='empty light wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.5,
        extra_negative='thick pad, stack of sticky notes, notepad block, cube, chopsticks, food, cup, pen, pencil, clutter',
        obj_key='lunchbox',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 도시락 완료")
    display(img)

print("🎉 [도시락] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-20] 책상 (desk) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-20] 책상 (desk) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['wide shot, angled 45-degree view', 'wide shot, eye-level front view', 'wide shot, angled 45-degree view']
PAPER_BASE = 'white A4 paper'

# 1) 메모 있는 사진 12장 (fakeC_desk_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    img = generate(
        obj='wooden study desk with a large white A4 paper lying on top of the desk',
        bg='wooden floor',
        light=random.choice(lights),
        paper=PAPER_BASE,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='chair, laptop, monitor, books, clutter, two desks, two notes',
        obj_key='desk',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 책상 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_desk_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single wooden study desk',
        bg='wooden floor',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='chair, laptop, monitor, books, clutter, two desks, two notes',
        obj_key='desk',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 책상 완료")
    display(img)

print("🎉 [책상] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-21] 마이크 (microphone) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-21] 마이크 (microphone) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'eye-level front view', 'eye-level side view']
PAPER_BASE = 'small yellow sticky note beside the microphone base'

# 1) 메모 있는 사진 12장 (fakeC_microphone_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='desktop microphone on a small stand',
        bg='wooden desk',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='',
        obj_key='microphone',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 마이크 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_microphone_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='desktop microphone on a small stand',
        bg='wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='',
        obj_key='microphone',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 마이크 완료")
    display(img)

print("🎉 [마이크] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-22] 목도리 (muffler) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-22] 목도리 (muffler) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'top-down view']
PAPER_BASE = 'bright yellow sticky note'

# 1) 메모 있는 사진 12장 (fakeC_muffler_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    img = generate(
        obj='folded dark navy knitted scarf with fringed ends and a bright yellow sticky note on it',
        bg='light wooden floor',
        light=random.choice(lights),
        paper=PAPER_BASE,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='person, neck, blanket, sweater, pen, pencil, two notes, clutter',
        obj_key='muffler',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 목도리 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_muffler_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='folded dark navy knitted scarf with fringed ends',
        bg='light wooden floor',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='person, neck, blanket, sweater, pen, pencil, two notes, clutter',
        obj_key='muffler',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 목도리 완료")
    display(img)

print("🎉 [목도리] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-23] 장갑 (gloves) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-23] 장갑 (gloves) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['top-down view', 'angled 45-degree view', 'eye-level front view']
PAPER_BASE = 'small yellow sticky note beside the gloves'

# 1) 메모 있는 사진 12장 (fakeC_gloves_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    paper = PAPER_BASE
    if "yellow sticky note" in paper and random.random() < 0.4:
        paper = paper.replace("yellow sticky note", "white memo paper")
    img = generate(
        obj='pair of winter gloves',
        bg='wooden table',
        light=random.choice(lights),
        paper=paper,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='',
        obj_key='gloves',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 장갑 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_gloves_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='pair of winter gloves',
        bg='wooden table',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=7.0,
        extra_negative='',
        obj_key='gloves',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 장갑 완료")
    display(img)

print("🎉 [장갑] 15장 완료! (메모 12 + 메모없음 3)")


#### 🚀 [4-24] 향수 (perfume) 15장 양산 (메모 12 + 메모없음 3)


In [ ]:
# [4-24] 향수 (perfume) 15장 양산 (메모 있음 12장 + 메모 없음 3장)
MEMO_COUNT = 12
NOMEMO_COUNT = 3

lights = ['soft window light', 'bright indoor fluorescent light', 'dim warm lamp light']
angles = ['angled 45-degree view', 'top-down view', 'angled 45-degree view']
PAPER_BASE = 'small white paper note'

# 1) 메모 있는 사진 12장 (fakeC_perfume_001_s{seed}.png ~ 012)
for i in range(MEMO_COUNT):
    img = generate(
        obj='single opaque black matte glass perfume bottle with spray cap',
        bg='dark wooden desk',
        light=random.choice(lights),
        paper=PAPER_BASE,
        angle=random.choice(angles),
        date=random.choice(DATES),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='yellow liquid, clear bottle, label, text on bottle, brand logo, thick pad, stack of notes, pen, two notes, clutter',
        obj_key='perfume',
        has_memo=True,
    )
    print(f"[메모 {i+1}/{MEMO_COUNT}] 향수 완료")
    display(img)

# 2) 메모 없는 사진 3장 (fakeC_perfume_nomemo_001_s{seed}.png ~ 003)
for i in range(NOMEMO_COUNT):
    img = generate(
        obj='single opaque black matte glass perfume bottle with spray cap',
        bg='dark wooden desk',
        light=random.choice(lights),
        angle=random.choice(angles),
        seed=random.randint(1, 99999),
        idx=i + 1,
        steps=35,
        guidance=6.5,
        extra_negative='yellow liquid, clear bottle, label, text on bottle, brand logo, thick pad, stack of notes, pen, two notes, clutter',
        obj_key='perfume',
        has_memo=False,
    )
    print(f"[메모없음 {i+1}/{NOMEMO_COUNT}] 향수 완료")
    display(img)

print("🎉 [향수] 15장 완료! (메모 12 + 메모없음 3)")


### 📌 [5번 셀] 생성 결과 압축 및 다운로드 (`fakeC.zip`)


In [ ]:
import glob
c_imgs = sorted(glob.glob(f"{IMG_DIR}/*.png"))
memo_cnt = sum(1 for p in c_imgs if "_nomemo_" not in os.path.basename(p))
nomemo_cnt = sum(1 for p in c_imgs if "_nomemo_" in os.path.basename(p))
print(f"📊 본 양산 이미지 총 {len(c_imgs)}장 (메모 있음 {memo_cnt}장 / 메모 없음 {nomemo_cnt}장)")

!cd "{OUT_DIR}" && zip -rq ../fakeC.zip C fake_mapping.csv
from google.colab import files
files.download("fakeC.zip")
print("📦 다운로드 완료! (C/ 폴더 + fake_mapping.csv)")
